In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata
from sklearn.preprocessing import LabelEncoder

In [ ]:
df_qs = pd.ExcelFile("University Ranking by Major 2025.xlsx")

# University Admission Prediction + QS-Aware Top-K Recommender

## Objetivo

	•	Predecir admitted (0/1) con features académicas + contexto.
	•	Enriquecer con qs_score por (university_clean, mapped_major).
	•	Resolver degree → mapped_major (1 a N) duplicando filas cuando aplique.
	•	Recomendación Top-K usando probabilidad del modelo + QS como señal auxiliar.

## Pipeline

### Imports

### Config + Paths

### Loaders

celda 1

In [9]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR

# Sube hasta encontrar la carpeta que contiene "src"
for _ in range(6):
    if (PROJECT_ROOT / "src").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_EXISTS:", (PROJECT_ROOT / "src").exists())
print("SRC_INIT_EXISTS:", (PROJECT_ROOT / "src" / "__init__.py").exists())

sys.path.insert(0, str(PROJECT_ROOT))
print("sys.path[0]:", sys.path[0])

NOTEBOOK_DIR: /Users/danimorera/Documents/Portfolio/FLRecomendator/notebooks
PROJECT_ROOT: /Users/danimorera/Documents/Portfolio/FLRecomendator
SRC_EXISTS: True
SRC_INIT_EXISTS: True
sys.path[0]: /Users/danimorera/Documents/Portfolio/FLRecomendator


celda 2

In [ ]:
from src.config import RAW_DIR, RANDOM_STATE
from src.utils import load_excel_safe, quick_profile

RAW_DIR, RANDOM_STATE

(PosixPath('/Users/danimorera/Documents/Portfolio/FLRecomendator/data/raw'),
 42)

celda 3

In [12]:
STUDENTS_PATH = RAW_DIR / "Five Lands Stats.xlsx"
QS_PATH = RAW_DIR / "University Ranking by Major 2025.xlsx"

STUDENTS_PATH.exists(), QS_PATH.exists()

(True, True)

celda 4

In [13]:
df_students = load_excel_safe(STUDENTS_PATH)
df_qs = load_excel_safe(QS_PATH)

quick_profile(df_students, "df_students")
quick_profile(df_qs, "df_qs")

=== df_students ===
shape: (664, 25)
columns: ['TYPE OF STUDENT', 'LAST YEAR INSTITUTION', 'SCHOOL\nCURRICULUM', 'SCHOOL\nGRADE', 'GPA', 'TOEFL \n(MAX)', 'SAT\n(SUPERSCORE)', 'PROFILE\n(4-16)', 'PEAK', 'Unnamed: 9', 'UNIVERSITY', 'COUNTRY', 'DEGREE', 'AREA OF STUDY', 'Unnamed: 14', 'ADMISSION', 'CURRENCY', 'TUITION\nYEAR', 'AID\nYEAR', 'AID\nYEAR (EUR)', 'AID\nYEAR (USD)', 'FINAL TUITION\nYEAR', 'FINAL TUITION\nYEAR (EUR)', 'FINAL TUITION\nYEAR (USD)', '% AID']

nulls (top 15):


Unnamed: 9                   1.000000
Unnamed: 14                  1.000000
PEAK                         0.947289
AID\nYEAR (USD)              0.887048
AID\nYEAR (EUR)              0.885542
AID\nYEAR                    0.873494
TUITION\nYEAR                0.362952
FINAL TUITION\nYEAR (EUR)    0.361446
FINAL TUITION\nYEAR (USD)    0.347892
FINAL TUITION\nYEAR          0.347892
SAT\n(SUPERSCORE)            0.250000
CURRENCY                     0.200301
ADMISSION                    0.105422
DEGREE                       0.021084
AREA OF STUDY                0.021084
dtype: float64


head:


,TYPE OF STUDENT,LAST YEAR INSTITUTION,SCHOOL\nCURRICULUM,SCHOOL\nGRADE,GPA,TOEFL \n(MAX),SAT\n(SUPERSCORE),PROFILE\n(4-16),PEAK,Unnamed: 9,UNIVERSITY,COUNTRY,DEGREE,AREA OF STUDY,Unnamed: 14,ADMISSION,CURRENCY,TUITION\nYEAR,AID\nYEAR,AID\nYEAR (EUR),AID\nYEAR (USD),FINAL TUITION\nYEAR,FINAL TUITION\nYEAR (EUR),FINAL TUITION\nYEAR (USD),% AID
0,FIRST YEAR,INTERNATIONAL SCHOOL OF TURIN,BASE 7,33.0,3.5,102,1000.0,11,NaN,NaN,AMSTERDAM UNIVERSITY OF APPLIED SCIENCE,NETHERLANDS,"Business, Communication, International Busines...",BUSINESS,NaN,YES,EUR,2300.0,NaN,NaN,NaN,2300.0,2300.0,2479.223033,0.000000
1,FIRST YEAR,INTERNATIONAL SCHOOL OF TURIN,BASE 7,33.0,3.5,102,1000.0,11,NaN,NaN,IE UNIVERSITY MADRID CAMPUS,SPAIN,"Business, Communication, International Busines...",BUSINESS,NaN,SELECTED,EUR,24000.0,NaN,NaN,NaN,24000.0,24000.0,25870.153388,0.000000
2,FIRST YEAR,INTERNATIONAL SCHOOL OF TURIN,BASE 7,33.0,3.5,102,1000.0,11,NaN,NaN,UNIVERSIDAD EUROPEA,SPAIN,"Business, Communication, International Busines...",BUSINESS,NaN,YES,EUR,14000.0,NaN,NaN,NaN,14000.0,14000.0,15090.922810,0.000000
3,FIRST YEAR,INTERNATIONAL SCHOOL OF TURIN,BASE 7,33.0,3.5,102,1000.0,11,NaN,NaN,SAINT LOUIS UNIVERSITY MADRID,SPAIN,"Business, Communication, International Busines...",BUSINESS,NaN,YES,EUR,24000.0,1600.0,1600.0,1724.676893,22400.0,22400.0,24145.476496,0.066667
4,FIRST YEAR,INTERNATIONAL SCHOOL OF TURIN,BASE 7,33.0,3.5,102,1000.0,11,NaN,NaN,CUNEF UNIVERSIDAD,SPAIN,"Business, Communication, International Busines...",BUSINESS,NaN,YES,EUR,11000.0,NaN,NaN,NaN,11000.0,11000.0,11331.000000,0.000000


=== df_qs ===
shape: (31, 10)
columns: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9']

nulls (top 15):


Unnamed: 4    1.000000
Unnamed: 6    1.000000
Unnamed: 8    1.000000
Unnamed: 0    0.967742
Unnamed: 2    0.935484
Unnamed: 3    0.741935
Unnamed: 5    0.677419
Unnamed: 7    0.677419
Unnamed: 9    0.483871
Unnamed: 1    0.419355
dtype: float64


head:


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9
0,QS World University Rankings By Subject 2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
